# Thermal-masked gain

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

DATA = Path("data")  # inputs:  fusion/data/<source>/rgb/ + ir/  (or visible.mp4 + thermal.mp4)
OUT = Path("out")    # outputs: fusion/out/<method>_<source>.mp4

SOURCE = "car_008"   # any folder name in data/
print("available:", sorted(p.name for p in DATA.iterdir() if p.is_dir()))


def pick(*candidates):
    return next(p for p in candidates if p.exists())


VISIBLE = pick(DATA / SOURCE / "visible.mp4", DATA / SOURCE / "rgb")
THERMAL = pick(DATA / SOURCE / "thermal.mp4", DATA / SOURCE / "ir")
OUTPUT = OUT / f"thermal_masked_gain_{SOURCE}.mp4"

SEQ_FPS = 30.0     # frame rate assumed for image-sequence sources
MAX_FRAMES = None  # set to a small number for a quick preview

## Parameters

- `THRESHOLD` – thermal intensity (0–1) at which the mask reaches 0.5.
- `SOFTNESS` – width of the ramp around the threshold; 0 gives a hard cut.
- `BLUR_KSIZE` – odd Gaussian kernel to feather the mask edges; 0 disables.
- `GAIN` – brightness multiplier where the mask is 1 (`1.0` → twice as bright).

In [ ]:
THRESHOLD = 0.74
SOFTNESS = 0.04
BLUR_KSIZE = 9
GAIN = 1.0

## Video I/O

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


class FrameSource:
    """Sequential frame reader over either a video file or a directory of images (sorted by name)."""

    def __init__(self, path):
        self.path = Path(path)
        self.is_dir = self.path.is_dir()
        if self.is_dir:
            self.files = sorted(p for p in self.path.iterdir() if p.suffix.lower() in IMAGE_EXTS)
            if not self.files:
                raise FileNotFoundError(f"no images in {self.path}")
        elif not self.path.is_file():
            raise FileNotFoundError(f"could not find video: {self.path}")

    def info(self):
        if self.is_dir:
            h, w = cv2.imread(str(self.files[0])).shape[:2]
            return dict(width=w, height=h, fps=SEQ_FPS, frame_count=len(self.files))
        cap = cv2.VideoCapture(str(self.path))
        if not cap.isOpened():
            raise FileNotFoundError(f"could not open video: {self.path}")
        try:
            return dict(
                width=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                height=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
                fps=float(cap.get(cv2.CAP_PROP_FPS)),
                frame_count=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
            )
        finally:
            cap.release()

    def __iter__(self):
        if self.is_dir:
            for f in self.files:
                yield cv2.imread(str(f))
            return
        cap = cv2.VideoCapture(str(self.path))
        try:
            while True:
                ok, frame = cap.read()
                if not ok:
                    return
                yield frame
        finally:
            cap.release()

    def read(self, index):
        if self.is_dir:
            return cv2.imread(str(self.files[index]))
        cap = cv2.VideoCapture(str(self.path))
        cap.set(cv2.CAP_PROP_POS_FRAMES, index)
        ok, frame = cap.read()
        cap.release()
        if not ok:
            raise IndexError(f"no frame {index} in {self.path}")
        return frame


def probe(path):
    return FrameSource(path).info()


def iter_frame_pairs(visible, thermal):
    """Yield (visible_bgr, thermal_bgr) pairs until either stream ends."""
    for frame_v, frame_t in zip(FrameSource(visible), FrameSource(thermal)):
        if frame_v.shape[:2] != frame_t.shape[:2]:
            raise ValueError(f"frame size mismatch: {frame_v.shape[:2]} vs {frame_t.shape[:2]}")
        yield frame_v, frame_t


def read_frame(path, index):
    return FrameSource(path).read(index)


info_v, info_t = probe(VISIBLE), probe(THERMAL)
assert (info_v["width"], info_v["height"]) == (info_t["width"], info_t["height"]), "resolution mismatch"
info_v, info_t

## Fusion steps

In [ ]:
def thermal_mask(thermal_bgr, threshold=THRESHOLD, softness=SOFTNESS, blur_ksize=BLUR_KSIZE):
    """Thermal frame -> float32 mask in [0, 1], 1 where the signal is strong."""
    gray = cv2.cvtColor(thermal_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0
    if softness <= 0:
        mask = (gray >= threshold).astype(np.float32)
    else:
        lo, hi = threshold - softness / 2, threshold + softness / 2
        mask = np.clip((gray - lo) / (hi - lo), 0.0, 1.0)
    if blur_ksize > 1:
        k = blur_ksize | 1
        mask = cv2.GaussianBlur(mask, (k, k), 0)
    return mask.astype(np.float32)


def brighten_by_mask(visible_bgr, mask, gain=GAIN):
    """Scale each pixel by (1 + gain * mask), clipped to 8-bit."""
    out = visible_bgr.astype(np.float32) * (1.0 + gain * mask[..., None])
    return np.clip(out, 0, 255).astype(np.uint8)


def fuse(visible_bgr, thermal_bgr):
    return brighten_by_mask(visible_bgr, thermal_mask(thermal_bgr))

## Preview a single frame

Check the thermal intensity distribution to pick `THRESHOLD`: it should sit between the background
and the hot spots.

In [ ]:
FRAME = 30
v, t = read_frame(VISIBLE, FRAME), read_frame(THERMAL, FRAME)
g = cv2.cvtColor(t, cv2.COLOR_BGR2GRAY) / 255.0
print("thermal percentiles 50/90/99/99.9/max:", [round(float(np.percentile(g, p)), 3) for p in (50, 90, 99, 99.9, 100)])

m = thermal_mask(t)
o = fuse(v, t)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, (title, img) in zip(axes.flat, [("visible", v), ("thermal", t), ("mask", m), ("fused", o)]):
    ax.imshow(img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2RGB), cmap="gray", vmin=0, vmax=1 if img.ndim == 2 else None)
    ax.set_title(title); ax.axis("off")
plt.tight_layout()

## Run over the whole video

In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
writer = cv2.VideoWriter(str(OUTPUT), cv2.VideoWriter_fourcc(*"mp4v"), info_v["fps"], (info_v["width"], info_v["height"]))
assert writer.isOpened(), f"could not open writer: {OUTPUT}"

total = min(info_v["frame_count"], info_t["frame_count"])
if MAX_FRAMES is not None:
    total = min(total, MAX_FRAMES)

written = 0
try:
    for frame_v, frame_t in iter_frame_pairs(VISIBLE, THERMAL):
        if MAX_FRAMES is not None and written >= MAX_FRAMES:
            break
        writer.write(fuse(frame_v, frame_t))
        written += 1
        if written % 50 == 0 or written == total:
            print(f"\r{written}/{total} frames", end="", flush=True)
finally:
    writer.release()
print(f"\nwrote {written} frames to {OUTPUT.resolve()}")